# 08 — Scenario Analysis

Raising the service level buys fewer stockouts with more inventory. The cost curve is U-shaped and its minimum is the economically reasonable service level — **under the stated assumptions**, which the data cannot supply.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

In [ ]:
import pandas as pd
from src.run import load_results
r = load_results(cfg)
sc = pd.DataFrame(r['scenarios']['scenario_table'])
sc.round(4)

## The analytical cross-check

Newsvendor theory says the optimal service level is the critical ratio `Cu / (Cu + Co)`. Computing both the closed form and the simulated optimum makes each a check on the other.

In [ ]:
r['scenarios']['newsvendor'], r['scenarios']['cost_optimal_service_level']

## Sensitivity: the recommendation moves with the assumptions

In [ ]:
pd.DataFrame(r['scenarios']['sensitivity']).round(4)

## The cost curve

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9,4))
x = (sc.service_level*100).astype(str)
ax.plot(x, sc.holding_cost_cu, 'o-', label='holding')
ax.plot(x, sc.stockout_cost_cu, 'o-', label='stockout')
ax.plot(x, sc.total_cost_cu, 'o-', lw=2.5, label='total')
ax.set_xlabel('service level (%)'); ax.set_ylabel('cost (CU)')
ax.legend(); ax.set_title('Overstock vs stockout trade-off')
plt.show()

## Example decision

The forecast turned into an order.

In [ ]:
from src.inference.forecaster import DemandForecaster, format_recommendation
from src.data.loader import test_fold
f = DemandForecaster.load(cfg)
seg = pd.read_parquet(cfg.resolve('data','processed_dir')/'segments.parquet')
sid = str(seg.sort_values('total_demand', ascending=False).index[0])
print(format_recommendation(f.recommend(panel, sid, origin=test_fold(cfg).origin)))